# Phase 3, Round 4 -- Before/after evaluation: base Qwen3-8B vs. fine-tuned (7-field)

Runs on **Colab or Kaggle free-tier GPU only** (same hardware contract as Phase 2). Evaluates the Round 4 adapter (trained on the 7-field target) over the same 140-example held-out set from `data/processed/eval.jsonl` -- **unchanged since v1**, the fixed comparison point across all four rounds.

## What's new in this notebook vs. the v1/v2/v3 version

1. **Reports the same original 4-field metrics** (component, defect_type, safety_risk, severity, all against `eval.jsonl`'s official labels) so v4 stays directly comparable to v1/v2/v3 -- this part of the metric logic is untouched.
2. **Plus new atomic-field metrics**: accuracy on `crash_described`/`fire_described`/`injury_described`, measured against `data/processed/eval_text_consistent.json` -- a REPORTING-ONLY side file with text-derived ground truth for these 3 new fields (`eval.jsonl` itself was never modified to add them).
3. **Plus a "text-consistent" accuracy number** for `safety_risk`/`severity`: the same predictions, graded against `eval_text_consistent.json`'s adjusted labels (5 of 140 rows corrected, same audit + hand-review methodology as the training data -- see `docs/label-strategy.md`'s Round 4 section) instead of the official ones. This is the "adjusted ceiling" number -- how well the model does against labels a human agrees with the text on, separate from the official cross-round-comparable number.
4. **Plus a relaxed `component` metric**: does the predicted component match ANY of the complaint's raw multi-value `COMPDESC` entries (bucketed through the same taxonomy), not just the first-listed one used as the strict single-label target. (`defect_type` doesn't get an equivalent relaxed metric -- it's derived from a single deterministic rule per complaint, not a multi-valued NHTSA field, so there's no analogous "co-occurring valid label" to check against.)

**Flagged assumptions (unchanged from v1-v3):**
1. Greedy decoding (`do_sample=False`), not sampling -- reproducibility over general chat-quality tuning.
2. Examples run one at a time, not batched -- avoids left-padding edge cases.

## Upload checklist

Bundle into **one** Kaggle Dataset -- any dataset name works, auto-detected:

- `data/processed/eval.jsonl` (140 rows, never changes between rounds)
- `data/processed/eval_text_consistent.json` (NEW for Round 4 -- the reporting-only adjusted-label side file)
- Only the **inference-relevant** v4 adapter files: `adapter_config.json`, `adapter_model.safetensors`, `tokenizer.json`, `tokenizer_config.json`, `chat_template.jinja`, `README.md`.

In [ ]:
%%capture
!pip install unsloth

## 1. Load the eval set

In [ ]:
import glob, json, os, shutil

def find_kaggle_dataset(*required_files):
    """Search /kaggle/input/*/ for a folder containing all of required_files --
    works no matter what the attached Dataset is named, so re-uploading under a new
    name never requires editing this notebook."""
    for candidate in sorted(glob.glob("/kaggle/input/*/")):
        if all(os.path.exists(os.path.join(candidate, f)) for f in required_files):
            return candidate.rstrip("/")
    return None

if not (os.path.exists("eval.jsonl") and os.path.exists("eval_text_consistent.json")):
    kaggle_dir = find_kaggle_dataset("eval.jsonl", "eval_text_consistent.json")
    if kaggle_dir:
        shutil.copy(os.path.join(kaggle_dir, "eval.jsonl"), "eval.jsonl")
        shutil.copy(os.path.join(kaggle_dir, "eval_text_consistent.json"), "eval_text_consistent.json")
        print(f"found and copied eval.jsonl + eval_text_consistent.json from {kaggle_dir}")
    else:
        try:
            from google.colab import files
            print("Upload data/processed/eval.jsonl and data/processed/eval_text_consistent.json:")
            files.upload()
        except ImportError:
            raise RuntimeError(
                "eval.jsonl/eval_text_consistent.json not found locally, and no "
                "/kaggle/input/*/ folder contains both. On Kaggle: attach a Dataset "
                "with both files -- any dataset name works, no path editing needed."
            )

with open("eval.jsonl", encoding="utf-8") as f:
    eval_rows = [json.loads(line) for line in f]
print(f"eval examples: {len(eval_rows)}")
assert len(eval_rows) == 140, f"expected 140 eval rows, got {len(eval_rows)} -- check the uploaded file"

with open("eval_text_consistent.json", encoding="utf-8") as f:
    text_consistent = json.load(f)
assert len(text_consistent) == 140, f"expected 140 entries in eval_text_consistent.json, got {len(text_consistent)}"
n_corrected = sum(1 for v in text_consistent.values()
                   if v["label_source"] in ("text_corrected_downgrade", "text_corrected_upgrade_handreviewed"))
print(f"text-consistent reporting layer loaded -- {n_corrected}/140 rows have an adjusted label")


## 2. Shared prompt format and JSON parsing

`SYSTEM_PROMPT` copied verbatim from `notebooks/train_qlora_dora.ipynb` -- must match training exactly, since a different system prompt at eval time would confound the before/after comparison for the fine-tuned model.

In [ ]:
import re

SYSTEM_PROMPT = (
    "You are an automotive safety complaint analyst. Given a raw consumer complaint "
    "about a vehicle, extract a structured JSON object with exactly these fields: "
    'component (string), defect_type (string), safety_risk ("yes" or "no"), '
    'severity ("low", "medium", or "high"), crash_described (true or false -- does '
    'the complaint text itself describe an actual collision/impact, not just '
    'mention a safety feature by name), fire_described (true or false -- does the '
    'text describe an actual fire/smoke/explosion event), injury_described (true '
    'or false -- does the text describe an actual injury to a person, not a '
    'hypothetical or averted one). Respond with only the JSON object.'
)

MAX_SEQ_LENGTH = 896  # matches train_qlora_dora_v4.ipynb -- raised from 768 for the 7-field target
MAX_NEW_TOKENS = 130  # 7-field JSON is longer than the 4-field one; generous headroom

def build_prompt(tokenizer, narrative):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Complaint:\n{narrative}"},
    ]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=False,
    )

_JSON_OBJ_PATTERN = re.compile(r"\{.*?\}", re.DOTALL)

def parse_json_output(raw_text):
    """Extract and parse the first {...} block. Returns (parsed_dict_or_None, raw_text).
    Never raises -- a model going off-script (extra prose, missing braces, etc.) is a
    JSON-validity failure to count, not a notebook crash."""
    match = _JSON_OBJ_PATTERN.search(raw_text)
    if not match:
        return None, raw_text
    try:
        obj = json.loads(match.group(0))
        if not isinstance(obj, dict):
            return None, raw_text
        return obj, raw_text
    except json.JSONDecodeError:
        return None, raw_text


## 3. Generation loop (shared by both models)

In [ ]:
import torch
from unsloth import FastLanguageModel

def run_eval(model, tokenizer, rows, label):
    FastLanguageModel.for_inference(model)  # Unsloth's native 2x-faster inference mode
    results = []
    for i, row in enumerate(rows):
        prompt_text = build_prompt(tokenizer, row["narrative"])
        inputs = tokenizer(prompt_text, return_tensors="pt").to("cuda")
        with torch.no_grad():
            output_ids = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False,          # greedy -- see flagged assumption #1 above
                pad_token_id=tokenizer.eos_token_id,
            )
        new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
        raw_output = tokenizer.decode(new_tokens, skip_special_tokens=True)
        parsed, raw = parse_json_output(raw_output)
        results.append({"odino": row["odino"], "parsed": parsed, "raw_output": raw})
        if (i + 1) % 20 == 0 or (i + 1) == len(rows):
            print(f"[{label}] {i + 1}/{len(rows)}")
    return results

## 4. Run the base model (zero-shot, no adapter)

In [ ]:
base_model, base_tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen3-8B-unsloth-bnb-4bit",
    max_seq_length = MAX_SEQ_LENGTH,
    load_in_4bit = True,
    dtype = None,
)

base_results = run_eval(base_model, base_tokenizer, eval_rows, "base")

## 5. Free GPU memory before loading the fine-tuned model

Loading both 8B (4-bit) models simultaneously would roughly double VRAM use for no reason -- evaluate sequentially instead.

In [ ]:
import gc

del base_model, base_tokenizer
gc.collect()
torch.cuda.empty_cache()

## 6. Run the fine-tuned model (base + LoRA/DoRA adapter)

In [ ]:
ADAPTER_DIR = find_kaggle_dataset("adapter_config.json", "adapter_model.safetensors")
if ADAPTER_DIR is None:
    raise RuntimeError(
        "No /kaggle/input/*/ folder found containing adapter_config.json + "
        "adapter_model.safetensors. Attach a Dataset with the adapter's inference files "
        "in it (see the upload checklist at the top) -- any dataset name works, no path "
        "editing needed."
    )
print(f"using adapter from: {ADAPTER_DIR}")

# EDIT THIS before each run: "epoch3" for the final/post-epoch-3 adapter, "epoch2" for
# the epoch-2 checkpoint. v4's eval loss regressed only 0.63% from epoch 2 to epoch 3
# (smaller than v1's 0.83% and v2's 0.79%) -- small enough that this round decides the
# checkpoint on actual downstream task metrics, not loss alone, so both get run and
# compared. This label only controls the output filename below, so running this
# notebook twice against the two checkpoints doesn't overwrite either result.
CHECKPOINT_LABEL = "epoch3"

finetuned_model, finetuned_tokenizer = FastLanguageModel.from_pretrained(
    model_name = ADAPTER_DIR,
    max_seq_length = MAX_SEQ_LENGTH,
    load_in_4bit = True,
    dtype = None,
)

finetuned_results = run_eval(finetuned_model, finetuned_tokenizer, eval_rows, "finetuned")


In [ ]:
# Fallback if the cell above fails to auto-attach the adapter (flagged assumption #2):
# from peft import PeftModel
# finetuned_model, finetuned_tokenizer = FastLanguageModel.from_pretrained(
#     model_name = "unsloth/Qwen3-8B-unsloth-bnb-4bit", max_seq_length = MAX_SEQ_LENGTH,
#     load_in_4bit = True, dtype = None,
# )
# finetuned_model = PeftModel.from_pretrained(finetuned_model, ADAPTER_DIR)
# finetuned_results = run_eval(finetuned_model, finetuned_tokenizer, eval_rows, "finetuned")

## 7. Metrics

Four parts: (a) the original 4-field metrics against `eval.jsonl`'s official labels, unchanged from v1-v3 for direct comparability; (b) new atomic-field metrics against `eval_text_consistent.json`; (c) "text-consistent" safety_risk/severity accuracy -- same predictions, graded against the adjusted labels instead of the official ones; (d) a relaxed `component` metric using the multi-value `component_raw` metadata.

In [ ]:
def norm(s):
    return (s or "").strip().upper() if isinstance(s, str) else ""

def field_correct(pred, actual_row, field):
    if pred is None:
        return False
    return norm(pred.get(field)) == norm(actual_row[field])

def bool_field_correct(pred, actual_bool, field):
    if pred is None or field not in pred or not isinstance(pred[field], bool):
        return False
    return pred[field] == actual_bool

# --- (a) original 4-field metrics, against eval.jsonl's OFFICIAL labels -----
# Unchanged logic from v1/v2/v3 -- this is what stays directly comparable across
# all four rounds. Do not add the 3 new fields into this function.
def compute_metrics(results, rows):
    n = len(rows)
    preds = [r["parsed"] for r in results]

    json_valid = sum(1 for p in preds if p is not None)
    metrics = {
        "n": n,
        "json_validity_rate": json_valid / n,
        "component_accuracy": sum(field_correct(p, r, "component") for p, r in zip(preds, rows)) / n,
        "defect_type_accuracy": sum(field_correct(p, r, "defect_type") for p, r in zip(preds, rows)) / n,
        "safety_risk_accuracy": sum(field_correct(p, r, "safety_risk") for p, r in zip(preds, rows)) / n,
        "severity_accuracy": sum(field_correct(p, r, "severity") for p, r in zip(preds, rows)) / n,
    }

    tp = sum(1 for p, r in zip(preds, rows) if r["safety_risk"] == "yes" and p is not None and norm(p.get("safety_risk")) == "YES")
    fp = sum(1 for p, r in zip(preds, rows) if r["safety_risk"] == "no" and p is not None and norm(p.get("safety_risk")) == "YES")
    fn = sum(1 for p, r in zip(preds, rows) if r["safety_risk"] == "yes" and not (p is not None and norm(p.get("safety_risk")) == "YES"))
    metrics["safety_risk_yes_precision"] = tp / (tp + fp) if (tp + fp) > 0 else None
    metrics["safety_risk_yes_recall"] = tp / (tp + fn) if (tp + fn) > 0 else None
    metrics["safety_risk_yes_tp_fp_fn"] = {"tp": tp, "fp": fp, "fn": fn}

    sev_labels = ["low", "medium", "high"]
    matrix = {a: {p: 0 for p in sev_labels + ["INVALID"]} for a in sev_labels}
    for p, r in zip(preds, rows):
        actual = r["severity"]
        pred_sev = norm(p.get("severity")).lower() if p is not None else None
        if pred_sev not in sev_labels:
            pred_sev = "INVALID"
        matrix[actual][pred_sev] += 1
    metrics["severity_confusion_matrix"] = matrix
    metrics["severity_per_tier_accuracy"] = {
        tier: matrix[tier][tier] / sum(matrix[tier].values()) if sum(matrix[tier].values()) > 0 else None
        for tier in sev_labels
    }
    return metrics


# --- (b) NEW: atomic-field metrics, against eval_text_consistent.json -------
def compute_atomic_metrics(results, rows, text_consistent):
    n = len(rows)
    preds = [r["parsed"] for r in results]
    metrics = {}
    for field in ["crash_described", "fire_described", "injury_described"]:
        correct = sum(
            bool_field_correct(p, text_consistent[r["odino"]][field], field)
            for p, r in zip(preds, rows)
        )
        metrics[f"{field}_accuracy"] = correct / n
    return metrics


# --- (c) NEW: "text-consistent" safety_risk/severity accuracy ("adjusted
# ceiling") -- same predictions, graded against eval_text_consistent.json's
# adjusted labels instead of eval.jsonl's official ones. Same shape as (a)'s
# safety_risk/severity metrics so the two numbers sit side by side cleanly.
def compute_text_consistent_metrics(results, rows, text_consistent):
    n = len(rows)
    preds = [r["parsed"] for r in results]
    adj = [text_consistent[r["odino"]] for r in rows]

    metrics = {
        "safety_risk_accuracy_text_consistent": sum(
            (norm(p.get("safety_risk")) if p else "") == norm(a["safety_risk"])
            for p, a in zip(preds, adj)
        ) / n,
        "severity_accuracy_text_consistent": sum(
            (norm(p.get("severity")) if p else "") == norm(a["severity"])
            for p, a in zip(preds, adj)
        ) / n,
    }

    tp = sum(1 for p, a in zip(preds, adj) if a["safety_risk"] == "yes" and p is not None and norm(p.get("safety_risk")) == "YES")
    fp = sum(1 for p, a in zip(preds, adj) if a["safety_risk"] == "no" and p is not None and norm(p.get("safety_risk")) == "YES")
    fn = sum(1 for p, a in zip(preds, adj) if a["safety_risk"] == "yes" and not (p is not None and norm(p.get("safety_risk")) == "YES"))
    metrics["safety_risk_yes_precision_text_consistent"] = tp / (tp + fp) if (tp + fp) > 0 else None
    metrics["safety_risk_yes_recall_text_consistent"] = tp / (tp + fn) if (tp + fn) > 0 else None
    return metrics


# --- (d) NEW: relaxed component metric using component_raw multi-value
# metadata -- does the predicted component match ANY of the complaint's raw
# COMPDESC entries (bucketed through the same taxonomy scripts/component_taxonomy.py
# uses), not just the first-listed one used as the strict single-label target.
# Inlined here (not imported) since this notebook is self-contained -- no repo
# file access on Kaggle/Colab beyond the uploaded Dataset.
_RAW_TO_BUCKET = {
    "ELECTRICAL SYSTEM": "ELECTRICAL SYSTEM", "POWER TRAIN": "POWER TRAIN",
    "ENGINE": "ENGINE", "ENGINE AND ENGINE COOLING": "ENGINE",
    "AIR BAGS": "AIR BAGS", "STEERING": "STEERING",
    "SERVICE BRAKES, HYDRAULIC": "BRAKES", "SERVICE BRAKES": "BRAKES",
    "SERVICE BRAKES, AIR": "BRAKES", "PARKING BRAKE": "BRAKES",
    "STRUCTURE": "STRUCTURE", "SUSPENSION": "SUSPENSION",
    "VEHICLE SPEED CONTROL": "VEHICLE SPEED CONTROL",
    "FUEL/PROPULSION SYSTEM": "FUEL SYSTEM", "FUEL SYSTEM, GASOLINE": "FUEL SYSTEM",
    "FUEL SYSTEM, OTHER": "FUEL SYSTEM", "EXTERIOR LIGHTING": "EXTERIOR LIGHTING",
    "VISIBILITY": "VISIBILITY", "VISIBILITY/WIPER": "VISIBILITY",
    "TIRES": "TIRES/WHEELS", "WHEELS": "TIRES/WHEELS",
    "SEAT BELTS": "SEAT BELTS", "SEATS": "SEATS",
    "FORWARD COLLISION AVOIDANCE": "ADAS/DRIVER ASSIST",
    "ELECTRONIC STABILITY CONTROL (ESC)": "ADAS/DRIVER ASSIST",
    "LANE DEPARTURE": "ADAS/DRIVER ASSIST", "BACK OVER PREVENTION": "ADAS/DRIVER ASSIST",
    "LATCHES/LOCKS/LINKAGES": "LATCHES/LOCKS/LINKAGES", "EQUIPMENT": "EQUIPMENT",
    "UNKNOWN OR OTHER": "OTHER",
}

def bucket_component(raw_top_level):
    return _RAW_TO_BUCKET.get(raw_top_level.strip().upper(), "OTHER")

def valid_component_set(component_raw):
    """component_raw is a comma-joined list of raw COMPDESC entries, each possibly
    colon-hierarchical (e.g. 'STRUCTURE:BODY') -- take the top-level segment of
    each, bucket it, and return the full set of valid bucketed labels for this
    (possibly multi-component) complaint."""
    entries = [e.split(":")[0].strip() for e in component_raw.split(",") if e.strip()]
    return {bucket_component(e) for e in entries}

def compute_relaxed_component_metric(results, rows):
    n = len(rows)
    preds = [r["parsed"] for r in results]
    correct = 0
    multi_component_n = 0
    for p, r in zip(preds, rows):
        valid_set = valid_component_set(r["component_raw"])
        if len(valid_set) > 1:
            multi_component_n += 1
        if p is not None and norm(p.get("component")) in {v.upper() for v in valid_set}:
            correct += 1
    return {
        "component_accuracy_relaxed_multivalue": correct / n,
        "multi_component_complaint_count": multi_component_n,
    }


base_metrics = compute_metrics(base_results, eval_rows)
finetuned_metrics = compute_metrics(finetuned_results, eval_rows)
base_metrics.update(compute_atomic_metrics(base_results, eval_rows, text_consistent))
finetuned_metrics.update(compute_atomic_metrics(finetuned_results, eval_rows, text_consistent))
base_metrics.update(compute_text_consistent_metrics(base_results, eval_rows, text_consistent))
finetuned_metrics.update(compute_text_consistent_metrics(finetuned_results, eval_rows, text_consistent))
base_metrics.update(compute_relaxed_component_metric(base_results, eval_rows))
finetuned_metrics.update(compute_relaxed_component_metric(finetuned_results, eval_rows))


In [ ]:
def pct(x):
    return f"{x:.1%}" if x is not None else "n/a"

print("=== (a) Original 4-field metrics, vs. eval.jsonl official labels (comparable to v1/v2/v3) ===")
print(f"{'metric':<28} {'base':>12} {'fine-tuned':>12}")
for key, label in [
    ("json_validity_rate", "JSON validity rate"),
    ("component_accuracy", "component accuracy"),
    ("defect_type_accuracy", "defect_type accuracy"),
    ("safety_risk_accuracy", "safety_risk accuracy"),
    ("severity_accuracy", "severity accuracy"),
]:
    print(f"{label:<28} {pct(base_metrics[key]):>12} {pct(finetuned_metrics[key]):>12}")

print()
print("safety_risk=yes precision/recall (official labels):")
print(f"{'':<28} {'base':>12} {'fine-tuned':>12}")
print(f"{'precision':<28} {pct(base_metrics['safety_risk_yes_precision']):>12} {pct(finetuned_metrics['safety_risk_yes_precision']):>12}")
print(f"{'recall':<28} {pct(base_metrics['safety_risk_yes_recall']):>12} {pct(finetuned_metrics['safety_risk_yes_recall']):>12}")

print()
print("severity per-tier accuracy (official labels):")
print(f"{'':<10} {'base':>12} {'fine-tuned':>12}")
for tier in ["low", "medium", "high"]:
    print(f"{tier:<10} {pct(base_metrics['severity_per_tier_accuracy'][tier]):>12} {pct(finetuned_metrics['severity_per_tier_accuracy'][tier]):>12}")

print()
print("fine-tuned severity confusion matrix (rows=actual official label, cols=predicted):")
print(f"{'':<8}" + "".join(f"{c:>9}" for c in ["low", "medium", "high", "INVALID"]))
for a in ["low", "medium", "high"]:
    row = finetuned_metrics["severity_confusion_matrix"][a]
    print(f"{a:<8}" + "".join(f"{row[c]:>9}" for c in ["low", "medium", "high", "INVALID"]))

print()
print("=== (b) NEW: atomic-field accuracy, vs. eval_text_consistent.json ===")
print(f"{'field':<24} {'base':>12} {'fine-tuned':>12}")
for field in ["crash_described", "fire_described", "injury_described"]:
    key = f"{field}_accuracy"
    print(f"{field:<24} {pct(base_metrics[key]):>12} {pct(finetuned_metrics[key]):>12}")

print()
print("=== (c) NEW: 'text-consistent' safety_risk/severity (adjusted-ceiling) ===")
print("Same predictions as (a), graded against eval_text_consistent.json instead of eval.jsonl official labels.")
print(f"{'metric':<38} {'base':>12} {'fine-tuned':>12}")
for key, label in [
    ("safety_risk_accuracy_text_consistent", "safety_risk accuracy"),
    ("severity_accuracy_text_consistent", "severity accuracy"),
    ("safety_risk_yes_precision_text_consistent", "safety_risk=yes precision"),
    ("safety_risk_yes_recall_text_consistent", "safety_risk=yes recall"),
]:
    print(f"{label:<38} {pct(base_metrics[key]):>12} {pct(finetuned_metrics[key]):>12}")
print()
print("Delta (fine-tuned): official vs. text-consistent safety_risk accuracy = "
      f"{pct(finetuned_metrics['safety_risk_accuracy'])} -> {pct(finetuned_metrics['safety_risk_accuracy_text_consistent'])}")

print()
print("=== (d) NEW: relaxed component accuracy (matches ANY co-occurring valid label) ===")
print(f"{'metric':<38} {'base':>12} {'fine-tuned':>12}")
print(f"{'component accuracy (strict, official)':<38} {pct(base_metrics['component_accuracy']):>12} {pct(finetuned_metrics['component_accuracy']):>12}")
print(f"{'component accuracy (relaxed)':<38} {pct(base_metrics['component_accuracy_relaxed_multivalue']):>12} {pct(finetuned_metrics['component_accuracy_relaxed_multivalue']):>12}")
print(f"(multi-component complaints in eval set: {finetuned_metrics['multi_component_complaint_count']} / {len(eval_rows)})")


## 8. Failure examples from the fine-tuned model

Up to 8 examples where the fine-tuned model got at least one field wrong. `guess_why` is a
data-grounded heuristic (checks the raw multi-component metadata and safety flags actually
present in the eval row), not a fabricated explanation -- treat it as a starting point for
the real error-analysis writeup in `docs/eval-report.md`, not a final verdict.

In [ ]:
def guess_why(row, pred):
    if pred is None:
        return "model did not produce parseable JSON"
    notes = []
    if not field_correct(pred, row, "component"):
        raw = row.get("component_raw", "")
        if "," in raw or ":" in raw:
            notes.append(f"multi/hierarchical component ('{raw}') -- model may have picked a different one than the first-listed primary label")
        else:
            notes.append("component mismatch on a single-component complaint -- possible taxonomy ambiguity")
    if not field_correct(pred, row, "defect_type"):
        notes.append("defect_type mismatch -- narrative may span more than one plausible defect category")
    if not field_correct(pred, row, "safety_risk"):
        direction = "missed a real safety signal" if row["safety_risk"] == "yes" else "over-flagged a non-risk complaint"
        notes.append(f"safety_risk miss (crash={row['crash']} fire={row['fire']} injured={row['injured']} deaths={row['deaths']}) -- {direction}")
        adj = text_consistent[row["odino"]]
        if adj["label_source"] in ("text_corrected_downgrade", "text_corrected_upgrade_handreviewed"):
            notes.append(f"NOTE: this row's official label was itself flagged as text-inconsistent -- "
                         f"text-consistent safety_risk is '{adj['safety_risk']}'")
    if not field_correct(pred, row, "severity"):
        notes.append("severity tier mismatch")
    for field in ["crash_described", "fire_described", "injury_described"]:
        adj = text_consistent[row["odino"]]
        if not bool_field_correct(pred, adj[field], field):
            notes.append(f"{field} mismatch (predicted={pred.get(field) if pred else None}, text-consistent={adj[field]})")
    return "; ".join(notes) if notes else "no field mismatch detected (check formatting/whitespace)"

failure_examples = []
for row, result in zip(eval_rows, finetuned_results):
    pred = result["parsed"]
    is_failure = pred is None or any(
        not field_correct(pred, row, f) for f in ["component", "defect_type", "safety_risk", "severity"]
    )
    if is_failure:
        failure_examples.append({
            "odino": row["odino"],
            "narrative": row["narrative"],
            "predicted": pred,
            "raw_output": result["raw_output"] if pred is None else None,
            "actual": {
                "component": row["component"], "defect_type": row["defect_type"],
                "safety_risk": row["safety_risk"], "severity": row["severity"],
            },
            "text_consistent": text_consistent[row["odino"]],
            "guess_why": guess_why(row, pred),
        })

failure_sample = failure_examples[:8]
print(f"{len(failure_examples)} total failing examples (original 4-field criteria); showing {len(failure_sample)}\n")
for ex in failure_sample:
    print(f"odino={ex['odino']}")
    print(f"narrative: {ex['narrative'][:200]}")
    print(f"predicted: {ex['predicted']}")
    print(f"actual:    {ex['actual']}")
    print(f"guess why: {ex['guess_why']}")
    print()


## 9. Save results

In [ ]:
OUTPUT_FILENAME = f"eval_results_v4_{CHECKPOINT_LABEL}.json"

output = {
    "eval_set_size": len(eval_rows),
    "checkpoint_label": CHECKPOINT_LABEL,
    "generation_config": {
        "do_sample": False,
        "max_new_tokens": MAX_NEW_TOKENS,
        "max_seq_length": MAX_SEQ_LENGTH,
        "enable_thinking": False,
    },
    "base": {
        "model": "unsloth/Qwen3-8B-unsloth-bnb-4bit (no adapter)",
        "metrics": base_metrics,
        "predictions": base_results,
    },
    "finetuned": {
        # dynamic, not a hardcoded epoch/eval_loss string -- that string went stale twice
        # across the v1/v2 reruns and had to be hand-corrected after the fact. The actual
        # adapter identity (which epoch, which eval_loss) lives in
        # docs/training-hyperparameters.md, cross-referenced by this ADAPTER_DIR path.
        "model": f"unsloth/Qwen3-8B-unsloth-bnb-4bit + adapter from {ADAPTER_DIR} ({CHECKPOINT_LABEL})",
        "metrics": finetuned_metrics,
        "predictions": finetuned_results,
        "failure_examples": failure_examples,
    },
}

with open(OUTPUT_FILENAME, "w", encoding="utf-8") as f:
    json.dump(output, f, ensure_ascii=False, indent=2)
print(f"saved {OUTPUT_FILENAME} -- download this and place it at eval/{OUTPUT_FILENAME} in the repo")
print("Remember: run this notebook a SECOND time with CHECKPOINT_LABEL set to the other "
      "checkpoint (cell 13) before comparing -- this round decides on both epoch2 and "
      "epoch3 results, not loss alone.")

try:
    from google.colab import files
    files.download(OUTPUT_FILENAME)
except ImportError:
    print(f"Not on Colab -- grab {OUTPUT_FILENAME} from the Kaggle notebook's output files panel instead.")


## Next

This round, the epoch2->epoch3 eval-loss regression was small (0.63%) -- smaller than
v1's (0.83%) and v2's (0.79%) -- small enough that it could be validation noise rather
than confirmed overfitting on a 140-example eval set. So the checkpoint decision isn't
made on loss alone this round: **run this notebook twice**, once with `CHECKPOINT_LABEL
= "epoch3"` (cell 13) against the already-downloaded final adapter, and once with
`CHECKPOINT_LABEL = "epoch2"` against the `outputs/checkpoint-114` adapter. Bring back
BOTH `eval_results_v4_epoch3.json` and `eval_results_v4_epoch2.json`
(`eval/eval_results_v4_epoch3.json`, `eval/eval_results_v4_epoch2.json`) -- whichever
wins on the real downstream task metrics (accuracy, safety_risk/severity precision-recall,
atomic-field accuracy) is the one that ships as v4, with both results and this reasoning
reported side by side in `docs/eval-report.md`'s Round 4 section.